In [1]:
import pandas as pd

import joblib
import sys
sys.path.append('../../')

import src.fda.kde.estimators           as kde

In [2]:
out_path    = "../../../densities4risk_doc/Tables/"

In [3]:
df_cv = joblib.load('../../data/processed/cv_divergences_long_20260316.jbl')
df_cv["forecast_kde_pp_params"] = df_cv["forecast_kde_params"] + "___" + df_cv["forecast_pp_params"]

In [4]:
df_cv

,kde_model,forecast_model,forecast_kde_params,forecast_pp_params,forecast_dFPC_dimension,forecast_VAR_q,date,KLD,JSD,L1_norm,L2_norm,LINF_norm,forecast_kde_pp_params
0,gaussian_rot_robust___False,gaussian_adaptive___rolling_w21___4___1,gaussian_adaptive,rolling_w21,4,1,2025-05-05,4.463318,0.259433,132390.399380,9300.290670,1423.011034,gaussian_adaptive___rolling_w21
1,gaussian_rot_robust___False,gaussian_adaptive___rolling_w21___4___1,gaussian_adaptive,rolling_w21,4,1,2025-05-06,4.747499,0.286736,136876.520476,9607.463644,1460.064834,gaussian_adaptive___rolling_w21
2,gaussian_rot_robust___False,gaussian_adaptive___rolling_w21___4___1,gaussian_adaptive,rolling_w21,4,1,2025-05-07,4.524809,0.260094,128365.689601,8981.680620,1361.856441,gaussian_adaptive___rolling_w21
3,gaussian_rot_robust___False,gaussian_adaptive___rolling_w21___4___1,gaussian_adaptive,rolling_w21,4,1,2025-05-08,6.517256,0.328244,153828.869188,10192.001729,1560.988364,gaussian_adaptive___rolling_w21
4,gaussian_rot_robust___False,gaussian_adaptive___rolling_w21___4___1,gaussian_adaptive,rolling_w21,4,1,2025-05-09,3.072839,0.173923,101820.998518,7167.118634,1072.201651,gaussian_adaptive___rolling_w21
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5580895,t-student_adaptive_df=5___rolling_w63,t-student_adaptive_df=3___rolling_w21___4___3,t-student_adaptive_df=3,rolling_w21,4,3,2025-08-21,0.333223,0.046379,48743.366429,5097.092268,1020.786493,t-student_adaptive_df=3___rolling_w21
5580896,t-student_adaptive_df=5___rolling_w63,t-student_adaptive_df=3___rolling_w21___4___3,t-student_adaptive_df=3,rolling_w21,4,3,2025-08-22,0.296705,0.041823,45993.948267,4825.992733,945.453217,t-student_adaptive_df=3___rolling_w21
5580897,t-student_adaptive_df=5___rolling_w63,t-student_adaptive_df=3___rolling_w21___4___3,t-student_adaptive_df=3,rolling_w21,4,3,2025-08-25,0.397947,0.054699,52575.805118,5515.372768,1100.757320,t-student_adaptive_df=3___rolling_w21
5580898,t-student_adaptive_df=5___rolling_w63,t-student_adaptive_df=3___rolling_w21___4___3,t-student_adaptive_df=3,rolling_w21,4,3,2025-08-26,0.363514,0.050627,51146.997399,5460.714541,1121.880012,t-student_adaptive_df=3___rolling_w21


In [5]:
df_cv.groupby(["forecast_kde_params", "forecast_pp_params"], as_index=False)[["KLD", "JSD", "L1_norm", "L2_norm", "LINF_norm"]].mean().sort_values(by="KLD").to_clipboard()

In [9]:
def structure_results_df(df):
    import pandas as pd
    
    # 1. Copy to avoid SettingWithCopy warnings
    results = df.copy()
    
    # 2. Extract Kernel
    results['kernel'] = results['forecast_kde_params'].str.split('_').str[0]
    
    # 3. Extract Degrees of Freedom (df)
    results['df'] = results['forecast_kde_params'].str.extract(r'df=(\d+)')
    
    # 4. Extract Method
    def extract_method(text):
        if 'cross_validate' in text:
            return 'cross_validate'
        elif 'adaptive' in text:
            return 'adaptive'
        elif 'rot_robust' in text:
            return 'rot_robust'
        elif 'rot' in text:
            return 'rot'
        else:
            return 'unknown'
    
    results['method'] = results['forecast_kde_params'].apply(extract_method)
    
    # 5. Build full method label
    results['method_full'] = results.apply(
        lambda x: f"{x['method']} (df={x['df']})" if pd.notnull(x['df']) else x['method'], 
        axis=1
    )
    
    # 6. Pretty labels (publication-friendly)
    def pretty_labels(text):
        mapping = {
            'cross_validate': 'Cross-validation',
            'adaptive': 'Adaptive',
            'rot_robust': 'ROT (robust)',
            'rot': 'ROT',
            'rolling_w5': 'Rolling (5 days)',
            'rolling_w21': 'Rolling (21 days)',
            'rolling_w63': 'Rolling (63 days)',
            'expanding': 'Expanding'
        }
        return mapping.get(text, text)
    
    results['method_full'] = results['method_full'].apply(pretty_labels)
    results['forecast_pp_params'] = results['forecast_pp_params'].apply(pretty_labels)
    
    # 7. Select and rename columns (🔥 FIX HERE)
    cols_to_keep = [
        'kernel', 'method_full', 'forecast_pp_params', 
        'KLD', 'JSD', 'L1_norm', 'L2_norm', 'LINF_norm'
    ]
    
    final_df = results[cols_to_keep].rename(columns={
        'method_full': 'bandwidth selection',
        'forecast_pp_params': 'preprocessing',
        'L1_norm': r'$L_1$',
        'L2_norm': r'$L_2$',
        'LINF_norm': r'$L_\infty$'
    })
    
    # 8. Aggregate
    final_df = final_df.groupby(
        ['kernel', 'bandwidth selection', 'preprocessing'],
        as_index=False
    )[["KLD", "JSD", r"$L_1$", r"$L_2$", r"$L_\infty$"]] \
     .mean() \
     .sort_values(by="KLD")
    
    # 9. LaTeX-safe strings (only for text columns)
    def latex_escape(text):
        if isinstance(text, str):
            return text.replace('_', r'\_')
        return text
    
    for col in ['kernel', 'bandwidth selection', 'preprocessing']:
        final_df[col] = final_df[col].apply(latex_escape)
    
    return final_df

cv_best_models = structure_results_df(df_cv).reset_index(drop=True)

In [12]:
cv_best_models.index = cv_best_models.index + 1

In [13]:
formatted_cv_summary = (   
    cv_best_models.iloc[:10,:]
    .style.background_gradient(
        cmap="RdYlGn_r",   # green = low, red = high
        axis=0            # column-wise
    )
    .format(precision=4)
)
formatted_cv_summary

,kernel,bandwidth selection,preprocessing,KLD,JSD,$L_1$,$L_2$,$L_\infty$
1,t-student,adaptive (df=3),Rolling (5 days),0.8080,0.0717,57668.2448,4405.4468,699.4913
2,t-student,cross\_validate (df=3),Rolling (5 days),0.8157,0.0727,57779.7239,4405.3903,701.5206
3,t-student,adaptive (df=4),Rolling (5 days),0.8227,0.0719,57756.5228,4414.8480,700.3932
4,t-student,cross\_validate (df=4),Rolling (5 days),0.8244,0.0738,58168.9438,4430.3680,703.1270
5,t-student,cross\_validate (df=5),Rolling (5 days),0.8293,0.0744,58426.6717,4447.9902,705.1816
6,t-student,adaptive (df=5),Rolling (5 days),0.8331,0.0721,57836.6671,4422.6877,701.2978
7,epanechnikov,ROT (robust),Rolling (5 days),0.8739,0.0736,58295.4459,4444.5761,698.8826
8,gaussian,Adaptive,Rolling (5 days),0.8849,0.0734,58367.0411,4471.5823,708.3319
9,epanechnikov,ROT,Rolling (5 days),0.8850,0.0774,59527.8723,4528.8880,710.4730
10,epanechnikov,Adaptive,Expanding,0.8940,0.0846,61459.6849,4633.2553,743.7473


In [14]:
formatted_cv_summary \
    .to_latex(
    buf     = '/'.join([out_path, "cv_results_code.tex"]),
    caption = "Results for expanding window cross-validation t+1 forecasts.",
    label   = "tab:cv_results",
    convert_css=True,
    # column_format="c", #* len(cv_results_summary.columns)
    hrules  = True
    )